# SPX calibration — piecewise-constant chaos

Run one calibration with the common hyperparameters matched to the current
`Real data/Legendre/Calibration.ipynb`: order 3, learning rate 1e-4, a
600-iteration cap, the same stopping and scheduler settings, sampling seeds,
Heston fitting/projection budgets, and zero regularization weights.

The piecewise time grid, tensor backend, Gauss-Hermite pricing at the first two
maturities, and quadrature nodes are retained. Both methods use plain Heston
projection. Monte Carlo budgets at the other maturities match
Legendre at those same maturities. Rates and dividends remain aligned.

Use **Restart Kernel → Run All**. Results are saved under `Statistics calibration/single/`.


SPX inputs and numerical results are not distributed. Supply your own data as described in the repository README.


## 1. Imports and data

Paths work from the notebook directory or the repository root. Rerunning this setup cell reloads the shared pricing module, so a previously imported class is not reused.

In [ ]:
from pathlib import Path
import hashlib
import importlib
import json
import pickle
import random
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import Markdown, display

ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "wiener_chaos/core/model.py").exists()),
    None,
)
if ROOT is None:
    raise FileNotFoundError("Open this notebook from within the project directory.")
STUDY_DIR = ROOT / "Real data" / "Piecewise constant"
NOTEBOOK_PATH = STUDY_DIR / "Calibration.ipynb"
if str(ROOT) in sys.path:
    sys.path.remove(str(ROOT))
sys.path.insert(0, str(ROOT))

from wiener_chaos.experiments.calibration import save_artifacts
from wiener_chaos.experiments.provenance import source_fingerprint
from wiener_chaos.pricing.options import compute_ivs
from wiener_chaos.core import chaos as chaos_module
chaos_module = importlib.reload(chaos_module)
from wiener_chaos.pricing import heston as heston_module
heston_module = importlib.reload(heston_module)
from wiener_chaos.core import model as pricing_module
pricing_module = importlib.reload(pricing_module)
ModelStock = pricing_module.ModelStock
print(f"Pricing module loaded: {pricing_module.__file__}")

DATA_PATH = ROOT / "Real data" / "real_data.pkl"
if not DATA_PATH.is_file():
    raise FileNotFoundError(
        "SPX inputs are private and are not distributed. Supply your own quotes "
        "and run Real data/Data.ipynb first; see the README input format."
    )
with DATA_PATH.open("rb") as handle:
    data = pickle.load(handle)

pd.set_option("display.max_columns", 12)
plt.rcParams.update({"figure.figsize": (10, 4), "axes.grid": True, "grid.alpha": 0.25})

## 2. Calibration settings

Common settings are copied from the current Legendre calibration notebook:

- Model: `d=2`, `P=3`; the retained grid has ten intervals.
- Heston: the same surface-derived guess, optimizer options, 640,000 projection
  paths, 201 time points, and one projection repetition.
- Optimizer: `lr=1e-4`, at most **600** steps, patience **60**, `tol_error=2e-8`,
  and `limit_error_above=2e-6`, with the same plateau scheduler.
- Sampling: refresh training paths every **50** steps; monitor training loss
  (`check_validation_set_MC=None`). Control-variate weights use all pricing
  samples, matching Legendre's omitted `n_samples_cv` setting.
- Both regularization weights are **zero**; moment control variates have degree 2.
- Tensor chunk size: **32,768**; print every **25** steps. Device selection,
  CPU fallback, base seed, and fixed comparison seeds match Legendre.

Basis-specific choices remain explicit. The first two maturities retain
Gauss-Hermite quadrature with **40** and **25** nodes, and no MC budgets. Subsequent
maturities use **400,000** paths for the next two and **200,000** for the
remaining six, both during optimization and independent evaluation. Each uses
**2 million** MC paths in total, alongside the two quadrature maturities.
Evaluation counts are copied from optimization counts; the samples use separate seeds.

The piecewise grid is preserved. Heston projection uses
`projection_control="none"` and `projection_antithetic=False`, matching Legendre. Projection path counts and repetitions are also identical.

The monitored objective is vega-weighted price MSE, while the reported accuracy
is independent-evaluation IV MAE. A run that reaches 600 steps is flagged for
convergence inspection. All quotes are fitted; fresh MC draws are not held-out
market observations.

### Optional heuristic-only Heston initialization

Leave `CONFIG["initialization"]["fit_heston"] = True` for the current fitted-Heston
warm start. Set it to `False` to use the existing surface-based Heston heuristic
directly, skipping Heston optimization. The chaos projection and subsequent chaos
calibration still run with the same budgets. `heston_fit` is empty when the Heston
optimizer is skipped. Use the same choice in calibration and calibration study.


In [ ]:
CUDA_DEVICE_INDEX = 3
if torch.cuda.is_available() and not 0 <= CUDA_DEVICE_INDEX < torch.cuda.device_count():
    raise RuntimeError(f"CUDA device {CUDA_DEVICE_INDEX} is unavailable.")
DEVICE = f"cuda:{CUDA_DEVICE_INDEX}" if torch.cuda.is_available() else "cpu"
if DEVICE == "cpu":
    torch.set_num_threads(1)

N_MATURITIES = len(data["maturities"])
CONFIG = {
    # Master seed for reproducible initialization and sampling.
    "base_seed": 42,
    # Device used for model tensors and pricing.
    "device": DEVICE,
    # Algorithm for evaluating conditional chaos moments.
    "backend": 'tensor',
    # Number of Brownian drivers.
    "d": 2,
    # Maximum total chaos order.
    "P": 3,
    # Time-grid endpoints defining the piecewise Brownian basis.
    "pi_M": [0.0, *data["maturities"]],
    # Paths evaluated per chunk by the tensor pricing kernel.
    "pricing_chunk_size": 32768,
    # Standard deviation for random initialization before normalization.
    "init_scale": 0.001,
    # How the starting chaos coefficients are obtained.
    "initialization": {
        # Choose a fitted-Heston projection or a random chaos start.
        "type": "calibrate_heston",
        # Starting point for the Heston parameter optimizer.
        "initial_guess": "surface",
        # False uses the surface heuristic directly, without Heston optimization.
        "fit_heston": True,
        # Plain projection estimator, matching Legendre.
        "projection_control": "none",
        # Match Legendre by disabling antithetic projection.
        "projection_antithetic": False,
        # Independent projection estimates averaged before normalization.
        "projection_repetitions": 1,
        # Brownian paths in each terminal-factor projection batch.
        "N": 5_000,
        # Number of projection batches; total paths are N times Ns.
        "Ns": 128,
        # Number of time-grid points in each Heston simulation.
        "n_steps_heston": 201,
        # Settings passed to the Heston optimizer.
        "options": {
            # Maximum Heston optimizer iterations.
            "maxiter": 500,
            # Heston optimizer tolerance for objective improvement.
            "ftol": 1e-12,
            # Heston optimizer tolerance for the gradient.
            "gtol": 1e-9,
        },
    },
    # Pricing allocations at calibration maturities.
    "pricing_calibr": {
        # Integration method for each calibration maturity (MC or Gauss-Hermite).
        "method": ["Gauss_Hermite"] * 2 + ["MC"] * 8,
        # MC paths per calibration maturity; None for quadrature.
        "N": [None, None, 400000, 400000, 200000, 200000, 200000, 200000, 200000, 200000],
        # Independent MC batches per calibration maturity.
        "Ns": [1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
        # Gauss-Hermite nodes per dimension; None for MC.
        "nodes": [40, 25] + [None] * 8,
    },
    # Optimizer, control-variate, and stopping settings.
    "calibration": {
        # Maximum number of optimizer steps.
        "n_iter": 600,
        # Initial AdamW learning rate.
        "lr": 1e-4,
        # Highest stock-price power used as an MC control variate.
        "poly_cv_degree": 2,
        # Non-improving monitoring checks allowed before early stopping.
        "patience": 60,
        # Minimum monitored improvement needed to reset patience.
        "tol_error": 2e-8,
        # Optimizer steps before activating MC control variates.
        "warmup_without_cv": 0,
        # Best monitored loss must be below this to stop early.
        "limit_error_above": 2e-6,
        # Steps between fresh training MC draws; None keeps draws fixed.
        "simulate_every": 50,
        # AdamW decoupled weight-decay coefficient.
        "reg_lambda": 0,
        # Weight of the penalty toward the starting chaos coefficients.
        "anchor_lambda": 0,
        # MC paths reserved to estimate control-variate weights.
        #"n_samples_cv": 10_000,
        # Steps between fixed validation checks; None monitors training loss.
        "check_validation_set_MC": None,
        # Steps between price-loss and speed reports; None suppresses reports.
        "print_loss_every_it": 25,
        # Whether model.calibrate displays fit plots.
        "print_plots": False,
        # Learning-rate scheduler settings for stalled monitoring loss.
        "plateau_kwargs": {
            # Whether lower or higher monitored values are better.
            "mode": "min",
            # Multiplier applied to the learning rate on a plateau.
            "factor": 0.95,
            # Non-improving checks before reducing the learning rate.
            "patience": 15,
            # Minimum change counted as scheduler improvement.
            "threshold": 1e-7,
            # Interpret threshold as absolute or relative.
            "threshold_mode": "abs",
            # Checks to wait after a learning-rate reduction.
            "cooldown": 0,
            # Lowest learning rate allowed by the scheduler.
            "min_lr": 1e-6,
            # Smallest learning-rate change worth applying.
            "eps": 1e-8,
        },
    },
}
# Use the same per-maturity path counts with independent evaluation draws.
CONFIG["comparison_N_per_mat"] = list(CONFIG["pricing_calibr"]["N"])
OUTPUT_DIR = STUDY_DIR / "Statistics calibration" / "single"
FIXED_VALIDATION_SEED = 4_000_042  # Reuse one independent validation sample.
FIXED_EVALUATION_SEED = 5_000_042  # Reuse one independent evaluation sample.
training_paths = sum(n or 0 for n in CONFIG["pricing_calibr"]["N"])
comparison_paths = sum(n or 0 for n in CONFIG["comparison_N_per_mat"])
print(f"Device: {DEVICE} | Single calibration | "
      f"Training MC paths: {training_paths:,} | Validation/evaluation MC paths: {comparison_paths:,}")
print(f"Anchor lambda: {CONFIG['calibration']['anchor_lambda']:.1e} | "
      f"Stopping tolerance: {CONFIG['calibration']['tol_error']:.1e} | "
      f"Initialization: per-run surface Heston + plain projection")
maturity_table = pd.DataFrame({
    "maturity": data["maturities"],
    "quotes": [len(row) for row in data["strikes"]],
    "rate": data["r_s"],
    "dividend_yield": data["q_s"],
    "pricing": CONFIG["pricing_calibr"]["method"],
    "training_N": CONFIG["pricing_calibr"]["N"],
    "validation_evaluation_N": CONFIG["comparison_N_per_mat"],
})
display(maturity_table)

## 3. Calibration routines

These cells define the workflow helpers. They do not start calibration.

### Reproducibility, saving, and error metrics

In [ ]:
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def json_safe(value):
    """Detach nested tensors and use JSON null for nonfinite diagnostics."""
    if torch.is_tensor(value):
        return json_safe(value.detach().cpu().tolist())
    if isinstance(value, np.ndarray):
        return json_safe(value.tolist())
    if isinstance(value, dict):
        return {str(k): json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(v) for v in value]
    if isinstance(value, (float, np.floating)):
        return float(value) if np.isfinite(value) else None
    if isinstance(value, np.integer):
        return int(value)
    return value


def save_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(json_safe(payload), indent=2, allow_nan=False) + "\n")
    temporary.replace(path)


def surface_metrics(actual, expected):
    """Do not silently discard invalid IVs or mismatched surface entries."""
    if len(actual) != len(expected):
        raise ValueError("Different numbers of maturity rows.")
    errors = []
    invalid = 0
    for row, reference in zip(actual, expected):
        if len(row) != len(reference):
            raise ValueError("Different numbers of strikes.")
        for value, target in zip(row, reference):
            if value is None or not np.isfinite(value) or not np.isfinite(target):
                invalid += 1
            else:
                errors.append(abs(float(value) - float(target)))
    return {
        "mae": float(np.mean(errors)) if errors and not invalid else None,
        "invalid_quotes": invalid,
    }

### Model setup


In [ ]:
def initialize_parameters_timed(model, **kwargs):
    """Time parameter initialization only, including completed GPU work."""
    if model.device.type == "cuda":
        torch.cuda.synchronize(model.device)
    init_started = time.perf_counter()
    initial = model.initialize_params(**kwargs)
    if model.device.type == "cuda":
        torch.cuda.synchronize(model.device)
    return initial, time.perf_counter() - init_started


def build_model(data, config):
    model = ModelStock(data["S0"],                        torch.device(config["device"]), torch.float32)
    grid = config["pi_M"]
    model.initialize_chaos(
        "piecewise_constant", grid[-1], config["d"], config["P"],
        len(grid) - 1, pi_M=grid, backend=config["backend"],
    )
    model.chaos.chunk_size = int(config["pricing_chunk_size"])
    if model.chaos.chunk_size < 1:
        raise ValueError("pricing_chunk_size must be positive.")
    # SPX rates are term structures: retain the rate paired with each maturity.
    model.set_up_for_Ts(data["maturities"], data["r_s"], data["q_s"])
    return model


## 4. Run one calibration

Initialize once, fit the market calls, and compute implied volatilities using each maturity's rate and dividend yield. The returned prices use fresh Monte Carlo samples after fitting. Loss and iteration speed are reported at the configured interval.

The complete numerical result is saved before plotting.


In [ ]:
progress = display(Markdown("Calibrating the SPX surface…"), display_id=True)
seed = CONFIG["base_seed"]
calibration_seed = seed + 3_000_000
seed_everything(seed)
model = build_model(data, CONFIG)
pricing = CONFIG["pricing_calibr"]
comparison_N = CONFIG["comparison_N_per_mat"]
if len(comparison_N) != len(data["maturities"]):
    raise ValueError("comparison_N_per_mat must cover every maturity.")
original_seeded_sample = model._simulate_X_block_with_seed

def fixed_comparison_sample(maturities_block, method_block, N_block, Ns_block, seed=None):
    if seed is None:
        return original_seeded_sample(maturities_block, method_block, N_block, Ns_block, seed)
    if len(maturities_block) != len(comparison_N):
        raise ValueError("This calibration uses all maturities in one block.")
    return original_seeded_sample(
        maturities_block, method_block, comparison_N, pricing["Ns"], seed,
    )

model._simulate_X_block_with_seed = fixed_comparison_sample
init_config = CONFIG["initialization"]
fit = None
if init_config["type"] == "calibrate_heston":
    progress.update(Markdown("Initializing Heston and projecting the terminal square-root factor…"))
    initial, init_seconds = initialize_parameters_timed(
        model,
        type="calibrate_heston", maturities=data["maturities"],
        strikes=data["strikes"], prices_market=data["prices"],
        r=data["r_s"], q=data["q_s"], seed=seed,
        **{key: value for key, value in init_config.items() if key not in ("type", "scale")},
    )
    fit = model.heston_fit_result
elif init_config["type"] == "random":
    initial, init_seconds = initialize_parameters_timed(model, type="random", scale=CONFIG["init_scale"])
else:
    raise ValueError(f"Unknown initialization type: {init_config['type']}")
# Calibration updates the parameter in place; preserve the actual starting vector.
initial_coefficients = initial.detach().cpu().clone()

progress.update(Markdown("Calibrating the chaos model…"))
# Keep the training random stream independent of the initialization choice.
seed_everything(calibration_seed)
d_a, prices_model, loss_ins, loss_oos, fit_seconds = model.calibrate(
    data["iv"], data["prices"], [data["maturities"]], data["strikes"],
    pricing["method"], pricing["N"], pricing["Ns"], pricing["nodes"],
    d_a=initial, validation_seed=FIXED_VALIDATION_SEED,
    evaluation_seed=FIXED_EVALUATION_SEED, **CONFIG["calibration"],
)
if not torch.isfinite(d_a).all():
    raise FloatingPointError("Nonfinite calibrated coefficients.")
iv_model = compute_ivs(prices_model, data["maturities"], data["strikes"],
                       data["S0"], data["r_s"], data["q_s"])
progress.update(Markdown("Saving calibration results…"))
monitor_name = ("best_training_price_mse" if CONFIG["calibration"]["check_validation_set_MC"] is None
                else "best_validation_price_mse")
result = json_safe({
    "config": CONFIG, "seed": seed, "calibration_seed": calibration_seed,
    "validation_seed": (FIXED_VALIDATION_SEED
                        if CONFIG["calibration"]["check_validation_set_MC"] is not None else None),
    "evaluation_seed": FIXED_EVALUATION_SEED,
    "training_N_per_mat": pricing["N"],
    "comparison_N_per_mat": comparison_N,
    "d_a": d_a, "initial_coefficients": initial_coefficients,
    "heston_init_params": getattr(model, "heston_params", None),
    "heston_fit": ({"success": bool(fit.success), "iterations": int(fit.nit),
                    "objective": float(fit.fun), "message": str(fit.message)}
                   if fit is not None else None),
    "maturities": data["maturities"], "r_s": data["r_s"], "q_s": data["q_s"],
    "loss_ins": loss_ins, "loss_oos": loss_oos,
    "calibration_iterations": model.last_calibration_iterations,
    monitor_name: model.last_calibration_monitor_loss,
    "prices_model": prices_model, "iv_model": iv_model,
    "iv_calibration_metrics": surface_metrics(iv_model, data["iv"]),
    "elapsed_seconds_init": init_seconds,
    "elapsed_seconds_chaos_calibration": fit_seconds,
    "elapsed_seconds_total": init_seconds + fit_seconds,
    "data_sha256": hashlib.sha256(DATA_PATH.read_bytes()).hexdigest(),
    "heston_source_sha256": hashlib.sha256(Path(heston_module.__file__).read_bytes()).hexdigest(),
    "model_source_sha256": hashlib.sha256(Path(pricing_module.__file__).read_bytes()).hexdigest(),
    "chaos_source_sha256": source_fingerprint(ROOT / "wiener_chaos/core/basis.py"),
    "notebook_source_sha256": hashlib.sha256(NOTEBOOK_PATH.read_bytes()).hexdigest(),
    "runtime": {"torch": torch.__version__, "numpy": np.__version__, "device": DEVICE},
})
save_json(OUTPUT_DIR / "result.json", result)
calibration_iv_mae = result["iv_calibration_metrics"]["mae"]
calibration_iv_text = (f"{calibration_iv_mae * 1e4:.2f} bp"
                       if calibration_iv_mae is not None else "unavailable (invalid IVs)")
progress.update(Markdown(
    f"Initialization: {result['elapsed_seconds_init']:.2f}s | "
    f"Calibration: {result['elapsed_seconds_chaos_calibration']:.2f}s | "
    f"Total: {result['elapsed_seconds_total']:.2f}s | "
    f"IV MAE: calibration {calibration_iv_text}"
))

if model.last_calibration_iterations >= CONFIG["calibration"]["n_iter"]:
    print("The iteration cap was reached; inspect convergence before interpreting this fit.")


## 5. Fit quality and runtime

IV errors are in basis points of volatility (`1 bp = 0.0001`). The bid/ask percentage counts fitted IVs inside the quoted interval. Nonfinite fitted values are counted separately and invalidate the IV MAE.

The smile grids retain the forward log-moneyness convention, `log(K/F_T)`, using the maturity-specific rate and dividend yield.

Timing reports parameter initialization, the `model.calibrate()` call, and their sum. Initialization includes the Heston fit and projection when selected; random initialization is typically negligible. Model setup, subsequent IV computation, held-out evaluation, and subsequent evaluation is excluded from the total. The calibration timer includes the pricing and reporting performed inside `model.calibrate()`.


In [ ]:
fit_rows = []
for T, fitted, target, bid, ask in zip(
    data["maturities"], iv_model, data["iv"], data["iv_bid"], data["iv_ask"],
):
    metrics = surface_metrics([fitted], [target])
    values = np.asarray(fitted, dtype=float)
    inside = np.isfinite(values) & (values >= np.asarray(bid)) & (values <= np.asarray(ask))
    fit_rows.append({
        "maturity": T, "quotes": len(values),
        "iv_mae_bp": None if metrics["mae"] is None else 1e4 * metrics["mae"],
        "invalid_iv_quotes": metrics["invalid_quotes"],
        "within_bid_ask": int(inside.sum()),
        "within_bid_ask_pct": 100 * inside.mean(),
    })
fit_by_maturity = pd.DataFrame(fit_rows).set_index("maturity")
metrics = result["iv_calibration_metrics"]
fit_summary = pd.DataFrame([{
    "quotes": int(fit_by_maturity["quotes"].sum()),
    "iv_mae_bp": None if metrics["mae"] is None else 1e4 * metrics["mae"],
    "invalid_iv_quotes": metrics["invalid_quotes"],
    "within_bid_ask_pct": 100 * fit_by_maturity["within_bid_ask"].sum() / fit_by_maturity["quotes"].sum(),
}], index=["SPX calibration"])
display(fit_summary.round(4))
display(fit_by_maturity.round(4))
print(
    f"Initialization: {result['elapsed_seconds_init']:.2f}s | "
    f"Calibration: {result['elapsed_seconds_chaos_calibration']:.2f}s | "
    f"Total: {result['elapsed_seconds_total']:.2f}s"
)


In [ ]:
def plot_spx_smiles(data, fitted_ivs):
    columns = 3
    rows = (len(data["maturities"]) + columns - 1) // columns
    fig, axes = plt.subplots(rows, columns, figsize=(13, 3.3 * rows), squeeze=False)
    for ax, T, strikes, market, fitted, bid, ask, r, q in zip(
        axes.flat, data["maturities"], data["strikes"], data["iv"], fitted_ivs,
        data["iv_bid"], data["iv_ask"], data["r_s"], data["q_s"],
    ):
        forward = data["S0"] * np.exp((r - q) * T)
        x = np.log(np.asarray(strikes) / forward)
        ax.fill_between(x, 100 * np.asarray(bid), 100 * np.asarray(ask),
                        alpha=.25, label="Market bid-ask")
        ax.plot(x, 100 * np.asarray(market), ".", label="Market IV")
        ax.plot(x, 100 * np.asarray(fitted, dtype=float), "-", label="Calibrated chaos")
        ax.set(title=f"T = {T:g}", xlabel="log(K / forward)", ylabel="Implied volatility (%)")
    for ax in list(axes.flat)[len(data["maturities"]):]:
        ax.set_visible(False)
    axes.flat[0].legend()
    fig.suptitle("SPX calibration against Market bid-ask")
    fig.tight_layout()
    plt.show()
    return fig

figures = {"calibration": plot_spx_smiles(data, iv_model)}


## 6. Saved results

`Statistics calibration/single/result.json` records the configuration, fitted
coefficients, prices, implied volatilities, seeds, timings, and provenance.
`coefficients/` contains `initial_coefficients.pt` and `calibrated_coefficients.pt`;
`tables/` contains `fit_summary.csv` and `fit_by_maturity.csv`. Comparison figures
are saved in `figures/` as PNG and PDF files.
Individual bid/ask smiles are saved in `figures/smiles/`.


In [ ]:
save_artifacts(
    OUTPUT_DIR,
    coefficients={"initial_coefficients": result["initial_coefficients"],
                  "calibrated_coefficients": result["d_a"]},
    tables={"fit_summary": fit_summary, "fit_by_maturity": fit_by_maturity},
    figures=figures,
)
print(f"Single-calibration results saved in: {OUTPUT_DIR}")

smile_dir = OUTPUT_DIR / "figures" / "smiles"
smile_dir.mkdir(parents=True, exist_ok=True)
model._plot_block_fit_final(
    [iv_model], data["maturities"], data["strikes"], input_type="iv",
    iv_mkt_block=None, iv_bid_block=data["iv_bid"], iv_ask_block=data["iv_ask"],
    labels=["Calibration"], save_folder=str(smile_dir), show_maturity_in_title=True,
)
print(f"Single-run results saved in: {OUTPUT_DIR}")
print(f"Bid/ask smile plots saved in: {smile_dir}")
